# Kidney CT Image Classification — PyTorch Gradio App

Run this locally from the project folder after training the PyTorch notebook. It loads `kidney_model.pth` from the local filesystem; it does not mount Google Drive or use Colab.

Expected: `kidney_model.pth` saved by `Chronic_kidney_disease_PyTorch.ipynb`. By default, the model file is expected in the current working directory. Update `MODEL_PATH` if it is saved elsewhere.


In [ ]:
# Install if needed:
# pip install torch torchvision gradio pillow fpdf2 numpy

from pathlib import Path
import datetime
import tempfile
import numpy as np
import torch
import torch.nn as nn
import gradio as gr
from PIL import Image
from torchvision import models, transforms
from fpdf import FPDF


In [ ]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
MODEL_PATH = Path("kidney_model.pth")
if not MODEL_PATH.exists():
    raise FileNotFoundError(
        f"Model checkpoint not found: {MODEL_PATH.resolve()}\n"
        "Train the model first, or set MODEL_PATH to the location of kidney_model.pth."
    )

checkpoint = torch.load(MODEL_PATH, map_location=DEVICE)
labels = checkpoint["class_names"]
IMAGE_SIZE = int(checkpoint.get("image_size", 224))

model = models.vgg16(weights=None)
model.classifier[6] = nn.Linear(model.classifier[6].in_features, len(labels))
model.load_state_dict(checkpoint["model_state_dict"])
model.to(DEVICE)
model.eval()

preprocess = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

print("Loaded checkpoint:", MODEL_PATH.resolve())
print("Classes:", labels)
print("Device:", DEVICE)


In [ ]:
def generate_report(image: Image.Image, predictions_dict: dict) -> str:
    pdf = FPDF()
    pdf.add_page()
    pdf.set_font("Arial", size=16)
    pdf.cell(190, 10, txt="Kidney CT Image Classification Report", ln=1, align="C")
    pdf.set_font("Arial", size=10)
    pdf.cell(190, 7, txt=f"Date: {datetime.date.today().strftime('%B %d, %Y')}", ln=1)
    pdf.ln(5)

    # FPDF expects a file path for the image; use a temporary PNG.
    with tempfile.NamedTemporaryFile(suffix=".png", delete=False) as tmp:
        temp_image_path = tmp.name
    image.convert("RGB").save(temp_image_path)
    pdf.image(temp_image_path, x=10, y=pdf.get_y(), w=90)
    pdf.ln(75)

    pdf.set_font("Arial", size=12, style="B")
    pdf.cell(190, 8, txt="Model output (confidence scores):", ln=1)
    pdf.set_font("Arial", size=10)
    for label, prob in predictions_dict.items():
        pdf.cell(190, 7, txt=f"{label}: {prob * 100:.2f}%", ln=1)

    pdf.ln(5)
    pdf.set_font("Arial", size=8)
    pdf.multi_cell(
        190, 5,
        txt=("Disclaimer: This tool is for educational/research use only. "
             "It is not a medical diagnostic device. Do not use these results "
             "to make healthcare decisions; consult a qualified medical professional.")
    )

    report_path = Path(tempfile.gettempdir()) / "kidney_classification_report.pdf"
    pdf.output(str(report_path))
    Path(temp_image_path).unlink(missing_ok=True)
    return str(report_path)


In [ ]:
def classify_and_report(image):
    if image is None:
        raise gr.Error("Please upload a CT image.")
    if isinstance(image, np.ndarray):
        pil_image = Image.fromarray(image.astype(np.uint8)).convert("RGB")
    else:
        pil_image = image.convert("RGB")

    tensor = preprocess(pil_image).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        logits = model(tensor)
        probabilities = torch.softmax(logits, dim=1)[0].cpu().numpy()

    result = {labels[i]: float(probabilities[i]) for i in range(len(labels))}
    pdf_path = generate_report(pil_image, result)
    return result, pdf_path

interface = gr.Interface(
    fn=classify_and_report,
    inputs=gr.Image(type="pil", label="Upload Kidney CT Image"),
    outputs=[
        gr.Label(num_top_classes=len(labels), label="Model confidence scores"),
        gr.File(label="Download report"),
    ],
    title="Kidney CT Image Condition Classification",
    description=(
        "Upload a CT image to view the model's predicted dataset class. "
        "For educational/research use only; not a medical diagnosis."
    ),
    flagging_mode="never",
)

# Local launch; share=False avoids publishing a public Gradio link by default.
interface.launch(share=False)
